# Module 1 — Medallion Architecture (Bronze / Silver / Gold)
Exam domain: **Data Modeling**

Databricks notebook version — uses widgets, `%sql`, and `DESCRIBE HISTORY`.

In [ ]:
dbutils.widgets.text("catalog", "hive_metastore")
dbutils.widgets.text("schema", "module1")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE {catalog}.{schema}")

## Bronze — raw ingestion

In [ ]:
from pyspark.sql import functions as F

raw_data = [
    (1, "Alice", "2024-01-01", "100.5"),
    (2, "Bob", "2024-01-02", "bad_value"),
    (3, None, "2024-01-03", "200.0"),
]
raw_df = spark.createDataFrame(raw_data, ["id", "name", "event_date", "amount_raw"])

bronze_df = (raw_df
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.lit("databricks_demo")))

bronze_df.write.format("delta").mode("overwrite").saveAsTable("bronze_events")
display(bronze_df)

## Silver — cleaned & validated

In [ ]:
bronze_df = spark.table("bronze_events")

silver_df = (bronze_df
    .withColumn("amount", F.col("amount_raw").cast("double"))
    .filter(F.col("amount").isNotNull())
    .filter(F.col("name").isNotNull())
    .withColumn("event_date", F.to_date("event_date"))
    .drop("amount_raw", "source_file"))

silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_events")
display(silver_df)

## Gold — business-level aggregates

In [ ]:
silver_df = spark.table("silver_events")

gold_df = (silver_df
    .groupBy("event_date")
    .agg(F.sum("amount").alias("total_amount"),
         F.count("*").alias("num_events")))

gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_daily_summary")
display(gold_df)

## DESCRIBE HISTORY — go through every field of the output

In [ ]:
%sql
DESCRIBE HISTORY gold_daily_summary